# Piper Vietnamese (sea-g2p) - ONNX Inference & Word-Level Subtitles
Notebook chạy inference từ file `.onnx` và `.onnx.json` (chạy rất nhẹ trên CPU hoặc GPU, hỗ trợ chọn giọng multi-speaker và xuất subtitle `.srt` / `.json` từng chữ).

In [ ]:
# 1. Cài đặt thư viện (nhẹ, không cần cài PyTorch)
import os, sys

REPO_URL = "https://github.com/18200245/piper-seag2p.git"
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
PIPER_DIR = os.path.join(WORK_DIR, "piper-seag2p")

if not os.path.exists(PIPER_DIR):
    !git clone -q {REPO_URL} {PIPER_DIR}

!pip install -q onnxruntime onnx sea-g2p

if os.path.join(PIPER_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(PIPER_DIR, "src"))

print("Ready! Piper src:", os.path.join(PIPER_DIR, "src"))

In [ ]:
# 2. Cấu hình đường dẫn file .onnx và .onnx.json
import glob

# Đặt đường dẫn trực tiếp hoặc để tự động tìm trong /kaggle / /content
MODEL_PATH  = ""  # Ví dụ: "/kaggle/working/piper_out/nhieugiong.onnx"
CONFIG_PATH = ""  # Để trống nếu file .onnx.json nằm cùng thư mục với tên <model>.onnx.json

if not MODEL_PATH or not os.path.exists(MODEL_PATH):
    cands = sorted(
        glob.glob("/kaggle/working/**/*.onnx", recursive=True)
        + glob.glob("/kaggle/input/**/*.onnx", recursive=True)
        + glob.glob("/content/**/*.onnx", recursive=True)
    )
    assert cands, "Không tìm thấy file .onnx! Hãy điền đường dẫn vào biến MODEL_PATH."
    MODEL_PATH = cands[0]

if not CONFIG_PATH:
    CONFIG_PATH = f"{MODEL_PATH}.json"
    if not os.path.exists(CONFIG_PATH):
        json_cands = sorted(
            glob.glob(os.path.join(os.path.dirname(MODEL_PATH), "*.json"))
            + glob.glob("/kaggle/working/**/*.onnx.json", recursive=True)
            + glob.glob("/kaggle/input/**/*.onnx.json", recursive=True)
            + glob.glob("/content/**/*.onnx.json", recursive=True)
        )
        assert json_cands, "Không tìm thấy file config .onnx.json! Hãy điền vào CONFIG_PATH."
        CONFIG_PATH = json_cands[0]

print("MODEL_PATH :", MODEL_PATH)
print("CONFIG_PATH:", CONFIG_PATH)

In [ ]:
# 3. Load model ONNX (bật include_alignments=True để lấy mốc thời gian từng âm/chữ)
from piper.voice import PiperVoice
from piper.config import SynthesisConfig

voice = PiperVoice.load(
    model_path=MODEL_PATH,
    config_path=CONFIG_PATH,
    use_cuda=False,             # Đổi thành True nếu đã cài onnxruntime-gpu và muốn chạy GPU
    include_alignments=True,    # Bắt buộc True nếu muốn xuất subtitle từng chữ
)

speaker_map = dict(voice.config.speaker_id_map or {})
id_to_speaker = {v: k for k, v in speaker_map.items()}
print(f"Sample rate: {voice.config.sample_rate} Hz | Num speakers: {voice.config.num_speakers}")
print("Danh sách speakers (tên -> speaker_id):", list(speaker_map.items())[:30])

In [ ]:
# 4. Đọc thử văn bản tiếng Việt và nghe trực tiếp
import wave
from IPython.display import Audio, display

TEXT = "Chương hai nghìn tám trăm mười. Bầy Gương. Xin chào, đây là giọng đọc tiếng Việt tổng hợp từ mô hình ONNX."
SPEAKER = 0          # Có thể điền tên speaker (vd: "anhuy") hoặc số ID (vd: 0)
LENGTH_SCALE = 1.0   # Tốc độ đọc: < 1.0 là nhanh hơn, > 1.0 là chậm hơn (vd: 0.95 hoặc 1.05)
NOISE_SCALE = 0.667  # Độ biểu cảm (mặc định 0.667)
NOISE_W_SCALE = 0.8  # Độ biến thiên nhịp điệu (mặc định 0.8)

if isinstance(SPEAKER, str):
    assert SPEAKER in speaker_map, f"Không tìm thấy speaker '{SPEAKER}'. Các speaker hợp lệ: {list(speaker_map.keys())[:20]}"
    speaker_id = speaker_map[SPEAKER]
else:
    speaker_id = int(SPEAKER) if voice.config.num_speakers > 1 else None

syn_config = SynthesisConfig(
    speaker_id=speaker_id,
    length_scale=LENGTH_SCALE,
    noise_scale=NOISE_SCALE,
    noise_w_scale=NOISE_W_SCALE,
)

out_wav = os.path.join(WORK_DIR, "infer_output.wav")
with wave.open(out_wav, "wb") as wav_file:
    voice.synthesize_wav(TEXT, wav_file, syn_config=syn_config)

spk_label = id_to_speaker.get(speaker_id, speaker_id)
print(f"Đã xuất: {out_wav} (speaker={spk_label})")
display(Audio(out_wav))

In [ ]:
# 5. Tổng hợp giọng đọc KÈM xuất Subtitle từng chữ (.srt và .json) từ Phoneme Alignments
import json as json_mod
import re
import wave
from sea_g2p import Normalizer

normalizer = Normalizer(lang="vi")
_PUNCT_STRIP = ".,?!:;\"“”'()[]{}…"

def format_srt_time(seconds: float) -> str:
    ms = int(round(max(0.0, seconds) * 1000))
    h, rem = divmod(ms, 3600_000)
    m, rem = divmod(rem, 60_000)
    s, ms = divmod(rem, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def synthesize_with_word_subtitles(
    text: str,
    wav_path: str,
    srt_path: str,
    json_path: str,
    syn_config: SynthesisConfig,
    pause_seconds: float = 0.2,
    words_per_cue: int = 1,  # Đặt = 1 để xuất subtitle từng chữ (Karaoke), hoặc 6-10 để gom theo cụm
):
    sr = voice.config.sample_rate
    pause_samples = int(pause_seconds * sr)
    pause_bytes = b"\x00\x00" * pause_samples

    # Tách câu giống VietnamesePhonemizer để khớp 1:1 với từng AudioChunk
    raw_sentences = [s.strip() for s in re.split(r"(?<=[.?!])\s+|\n+", text) if s.strip()]
    norm_sentences = [normalizer.normalize(s) for s in raw_sentences]

    word_timings = []
    cur_sample = 0

    with wave.open(wav_path, "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(sr)

        for idx, norm_sent in enumerate(norm_sentences):
            words = [w for w in norm_sent.split() if w.strip(_PUNCT_STRIP)]
            for chunk in voice.synthesize(norm_sent, syn_config=syn_config):
                wav_file.writeframes(chunk.audio_int16_bytes)
                chunk_total_samples = len(chunk.audio_int16_array)

                if chunk.phoneme_alignments:
                    # Tách thời lượng từng từ theo phoneme khoảng trắng " "
                    spans = []
                    w_start = cur_sample
                    w_samples = 0
                    for al in chunk.phoneme_alignments:
                        if al.phoneme == " ":
                            if w_samples > 0:
                                spans.append((w_start, w_start + w_samples))
                            w_start = w_start + w_samples + al.num_samples
                            w_samples = 0
                        else:
                            w_samples += al.num_samples
                    if w_samples > 0:
                        spans.append((w_start, w_start + w_samples))

                    for w_idx, word in enumerate(words):
                        if w_idx < len(spans):
                            s_samp, e_samp = spans[w_idx]
                        elif spans:
                            s_samp, e_samp = spans[-1]
                        else:
                            s_samp, e_samp = cur_sample, cur_sample + chunk_total_samples
                        word_timings.append({
                            "word": word,
                            "start": round(s_samp / sr, 3),
                            "end": round(e_samp / sr, 3),
                        })

                cur_sample += chunk_total_samples

            if idx < len(norm_sentences) - 1 and pause_samples > 0:
                wav_file.writeframes(pause_bytes)
                cur_sample += pause_samples

    # Ghi JSON word-level
    with open(json_path, "w", encoding="utf-8") as jf:
        json_mod.dump(word_timings, jf, ensure_ascii=False, indent=2)

    # Ghi SRT (từng chữ nếu words_per_cue=1, hoặc theo cụm)
    srt_Blocks = []
    for i in range(0, len(word_timings), max(1, words_per_cue)):
        group = word_timings[i : i + max(1, words_per_cue)]
        start_t = format_srt_time(group[0]["start"])
        end_t = format_srt_time(group[-1]["end"])
        cue_text = " ".join(item["word"] for item in group)
        srt_Blocks.append(f"{len(srt_Blocks) + 1}\n{start_t} --> {end_t}\n{cue_text}\n")

    with open(srt_path, "w", encoding="utf-8") as sf:
        sf.write("\n".join(srt_Blocks))

    return word_timings

sub_wav  = os.path.join(WORK_DIR, "infer_with_sub.wav")
sub_srt  = os.path.join(WORK_DIR, "infer_with_sub.srt")
sub_json = os.path.join(WORK_DIR, "infer_with_sub.json")

timings = synthesize_with_word_subtitles(
    TEXT,
    wav_path=sub_wav,
    srt_path=sub_srt,
    json_path=sub_json,
    syn_config=syn_config,
    words_per_cue=1,  # 1 = subtitle từng chữ; đổi thành 6 nếu muốn 6 chữ / dòng phụ đề
)

print(f"Đã lưu: {sub_wav}, {sub_srt}, {sub_json}")
for item in timings[:20]:
    print(f"[{item['start']:6.3f}s -> {item['end']:6.3f}s] {item['word']}")
display(Audio(sub_wav))